# Activity 9 — Neural Networks: Classifying Heart Disease Risk
## MRHA CardioWatch Program

**Course:** Introduction to Artificial Intelligence  
**Sessions:** 25, 26, 27, and 28  
**Libraries:** NumPy · Pandas · Matplotlib · scikit-learn · ipywidgets

---

## Background

Cardiovascular disease is the leading cause of death globally, responsible for an estimated **17.9 million deaths per year** (WHO). Early identification of at-risk individuals enables timely intervention — lifestyle counselling, medication, or closer monitoring — before a cardiac event occurs.

The **Maplewood Regional Health Authority (MRHA)** is expanding its *CardioWatch* program. In Activity 7 you built regression models to predict a patient's systolic blood pressure. Now the clinical team has a different question:

> **Will this patient develop heart disease within the next 5 years? (Yes / No)**

This is a **binary classification** problem. Instead of predicting a continuous number, the model must output one of two labels: `0` (no heart disease) or `1` (heart disease predicted).

You will use **Artificial Neural Networks (ANNs)**, specifically the **Multi-Layer Perceptron (MLP)** classifier, to solve this problem.

---

## Dataset

| Column | Description | Units |
|--------|-------------|-------|
| `age` | Patient age | years |
| `bmi` | Body Mass Index | kg/m² |
| `sbp` | Systolic Blood Pressure | mmHg |
| `cholesterol` | Total cholesterol level | mg/dL |
| `resting_hr` | Resting heart rate | bpm |
| `heart_disease` | Developed heart disease within 5 years (**target**) | 0 = No, 1 = Yes |

**Clinical reference ranges:**

| Feature | Low Risk | Elevated Risk |
|---------|----------|---------------|
| SBP | < 130 mmHg | ≥ 130 mmHg |
| Cholesterol | < 200 mg/dL | ≥ 200 mg/dL |
| Resting HR | 60–100 bpm | > 100 bpm |
| BMI | 18.5–24.9 kg/m² | ≥ 25 kg/m² |

---

## Assignment Structure

| Task | Model | Focus |
|------|-------|-------|
| **Task 1** | Simple Neural Network (1 hidden layer) | MCP neuron intuition → EDA → Training → Evaluation |
| **Task 2** | Deeper Network + Debugging | Overfitting · Regularization · Cross-validation · ROC |

Each task follows the same pipeline:
1. Conceptual foundation
2. Exploratory Data Analysis
3. Feature Normalization
4. Train / Test Split
5. Build and Train the Model
6. Evaluate the Model
7. Visualize Results
8. Interactive Experiment with ipywidgets

---
## Setup — Import Libraries

Run this cell first. It imports every library used throughout the notebook.

- **NumPy** — numerical arrays and math operations
- **Pandas** — tabular data (DataFrames)
- **Matplotlib** — plots and visualizations
- **scikit-learn** — machine learning models, metrics, and pipelines
- **ipywidgets** — interactive sliders inside the notebook

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import (
    accuracy_score, confusion_matrix,
    classification_report, roc_curve, auc
)
from sklearn.pipeline import Pipeline
import sklearn

import ipywidgets as widgets
from ipywidgets import interact

import warnings
warnings.filterwarnings('ignore')

np.set_printoptions(precision=3)
pd.options.display.float_format = '{:.2f}'.format

print('Libraries loaded successfully.')
print(f'  numpy      : {np.__version__}')
print(f'  pandas     : {pd.__version__}')
print(f'  sklearn    : {sklearn.__version__}')
print(f'  ipywidgets : {widgets.__version__}')

---
## Generate the MRHA Dataset

The cell below generates the synthetic patient dataset. The probability of developing heart disease follows a **logistic function** of five clinical variables:

$$\text{risk} = -1 + 0.04(\text{age}-50) + 0.05(\text{bmi}-25) + 0.02(\text{sbp}-130) + 0.005(\text{chol}-200) + 0.01(\text{hr}-70)$$

$$P(\text{heart disease}) = \frac{1}{1 + e^{-\text{risk}}}$$

The binary label is then drawn by comparing a uniform random number to this probability. This mirrors how logistic models are used in real epidemiological studies (e.g., the Framingham Heart Study).

> **Do not modify this cell.** The fixed random seed (`9`) ensures every student works with the same 300 patients.

In [ ]:
# -------------------------------------------------------
# MRHA CardioWatch Dataset  —  DO NOT MODIFY
# -------------------------------------------------------
np.random.seed(9)
N = 300

age         = np.random.uniform(30, 75, N)
bmi         = np.random.uniform(18.5, 38.0, N)
sbp         = np.random.normal(140, 25, N);         sbp         = np.clip(sbp,         100, 200)
cholesterol = np.random.normal(220, 40, N);         cholesterol = np.clip(cholesterol, 150, 320)
resting_hr  = np.random.normal(75, 12, N);          resting_hr  = np.clip(resting_hr,   50, 110)

risk = (
    -1
    + 0.04  * (age         - 50)
    + 0.05  * (bmi         - 25)
    + 0.02  * (sbp         - 130)
    + 0.005 * (cholesterol - 200)
    + 0.01  * (resting_hr  - 70)
)
prob          = 1 / (1 + np.exp(-risk))
heart_disease = (np.random.uniform(0, 1, N) < prob).astype(int)

df = pd.DataFrame({
    'age':         np.round(age, 1),
    'bmi':         np.round(bmi, 1),
    'sbp':         np.round(sbp, 1),
    'cholesterol': np.round(cholesterol, 1),
    'resting_hr':  np.round(resting_hr, 1),
    'heart_disease': heart_disease
})

print(f'Dataset: {df.shape[0]} patients x {df.shape[1]} variables')
print()
print('First 8 rows:')
display(df.head(8))
print()
print('Summary statistics:')
display(df.describe())

---
---
# TASK 1 — Simple Neural Network (One Hidden Layer)

In this task we build intuition for how neurons work, explore the data, preprocess it correctly, and train a two-hidden-layer MLP classifier. We then evaluate performance using clinical metrics.

### Conceptual foundation — from MCP neuron to MLP

The **McCulloch–Pitts (MCP) neuron** (Session 25) is the mathematical ancestor of every modern neural network. It computes a weighted sum of binary inputs and fires (`1`) if the sum exceeds a threshold:

$$\text{output} = \begin{cases} 1 & \text{if } \sum_i w_i x_i \geq \theta \\ 0 & \text{otherwise} \end{cases}$$

A **Multi-Layer Perceptron** stacks many such neurons into layers. Each hidden unit applies a non-linear **activation function** (ReLU by default in scikit-learn) instead of a hard threshold. During training, **backpropagation** (Session 26) computes gradients of the loss with respect to every weight, and gradient descent updates the weights to minimise the loss.

The architecture we will use for Task 1:

```
Input (5 features) → Hidden Layer 1 (100 neurons, ReLU)
                   → Hidden Layer 2  (50 neurons, ReLU)
                   → Output (1 neuron, Sigmoid → probability)
```

---
### Step 1.1 — MCP Neuron Demo: AND and OR Gates

Before training any model, let's build intuition by **manually implementing** two classic logic gates as MCP neurons (Session 25).

An MCP neuron with two binary inputs $x_1, x_2$ and weights $w_1, w_2$ fires when:
$$w_1 x_1 + w_2 x_2 \geq \theta$$

**AND gate:** fires only when *both* inputs are 1 → use $w_1 = w_2 = 1$, $\theta = 2$  
**OR gate:** fires when *at least one* input is 1 → use $w_1 = w_2 = 1$, $\theta = 1$

**Clinical analogy:** Imagine a simple rule:
- AND gate → flag only if *age > 60* **AND** *cholesterol > 240*
- OR gate  → flag if *age > 60* **OR** *cholesterol > 240*

The AND gate is more conservative (fewer flags, fewer false positives); the OR gate is more sensitive (more flags, fewer missed cases). A deep neural network learns much more complex, non-linear combinations of all five features simultaneously.

In [ ]:
def mcp_neuron(x1, x2, w1, w2, theta):
    '''MCP neuron: returns 1 if weighted sum >= threshold, else 0.'''
    return int(w1 * x1 + w2 * x2 >= theta)

# All possible binary inputs
inputs = [(0, 0), (0, 1), (1, 0), (1, 1)]

# AND gate: w1=1, w2=1, theta=2
and_results = [mcp_neuron(x1, x2, 1, 1, 2) for x1, x2 in inputs]

# OR gate: w1=1, w2=1, theta=1
or_results  = [mcp_neuron(x1, x2, 1, 1, 1) for x1, x2 in inputs]

print('MCP Neuron — Truth Tables')
print()
print(f'{"x1":>4} {"x2":>4} | {"AND (theta=2)":>14} | {"OR (theta=1)":>13}')
print('-' * 45)
for (x1, x2), a, o in zip(inputs, and_results, or_results):
    print(f'{x1:>4} {x2:>4} | {a:>14} | {o:>13}')

print()
print('Clinical analogy (x1 = high age, x2 = high cholesterol):')
print('  AND gate: flag patient ONLY if BOTH risk factors are present (conservative)')
print('  OR gate : flag patient if EITHER risk factor is present (sensitive)')
print()

# Visualise the decision regions
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
titles  = ['AND Gate  (theta = 2)', 'OR Gate  (theta = 1)']
results = [and_results, or_results]
colors  = {0: 'steelblue', 1: 'firebrick'}
labels  = {0: 'No fire (0)', 1: 'Fires  (1)'}

for ax, title, res in zip(axes, titles, results):
    for (x1, x2), r in zip(inputs, res):
        ax.scatter(x1, x2, color=colors[r], s=300, zorder=3,
                   edgecolors='black', linewidth=1.2)
        ax.text(x1 + 0.05, x2 + 0.05, str(r), fontsize=12, fontweight='bold')
    ax.set_xlim(-0.4, 1.4)
    ax.set_ylim(-0.4, 1.4)
    ax.set_xticks([0, 1])
    ax.set_yticks([0, 1])
    ax.set_xticklabels(['x1=0 (low age)', 'x1=1 (high age)'], fontsize=9)
    ax.set_yticklabels(['x2=0 (low chol)', 'x2=1 (high chol)'], fontsize=9)
    ax.set_title(title, fontsize=12)
    ax.set_xlabel('Input x1 (age risk)', fontsize=10)
    ax.set_ylabel('Input x2 (cholesterol risk)', fontsize=10)
    # Legend patches
    for val, col in colors.items():
        ax.scatter([], [], color=col, s=100, label=labels[val])
    ax.legend(fontsize=9, loc='lower right')

plt.suptitle('Step 1.1 — MCP Neuron: AND vs OR Gate', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

---
### Step 1.2 — Exploratory Data Analysis

Before building any model we must understand the data. We will:

1. Plot the **class distribution** — is the dataset balanced between healthy (0) and at-risk (1) patients?
2. Plot **histograms of each feature, coloured by class** — do any features clearly separate the two groups?

A highly imbalanced dataset (e.g., 90% class 0 / 10% class 1) can mislead a classifier into ignoring the minority class, producing high apparent accuracy but poor clinical sensitivity.

**Session 28 connection:** EDA is the first step in the ML debugging workflow — understanding data quality and class balance before touching any model.

In [ ]:
features = ['age', 'bmi', 'sbp', 'cholesterol', 'resting_hr']
class_counts = df['heart_disease'].value_counts().sort_index()
class_labels = ['No Heart Disease (0)', 'Heart Disease (1)']
class_colors = ['steelblue', 'firebrick']

fig = plt.figure(figsize=(16, 10))
gs  = gridspec.GridSpec(2, 6, figure=fig)

# --- Class distribution bar chart ---
ax_bar = fig.add_subplot(gs[0, :2])
bars = ax_bar.bar(class_labels, class_counts.values,
                  color=class_colors, edgecolor='white', width=0.5)
for bar, count in zip(bars, class_counts.values):
    ax_bar.text(bar.get_x() + bar.get_width() / 2,
                bar.get_height() + 2, str(count),
                ha='center', va='bottom', fontsize=11, fontweight='bold')
ax_bar.set_ylabel('Number of Patients', fontsize=11)
ax_bar.set_title('Class Distribution', fontsize=12)
ax_bar.set_ylim(0, max(class_counts.values) * 1.15)
pct = class_counts[1] / class_counts.sum() * 100
ax_bar.text(0.5, 0.92, f'{pct:.1f}% positive', ha='center',
            transform=ax_bar.transAxes, fontsize=10, color='firebrick')

# --- Feature histograms coloured by class ---
feat_positions = [
    gs[0, 2:4], gs[0, 4:6],
    gs[1, 0:2], gs[1, 2:4], gs[1, 4:6]
]
feat_titles = [
    'Age (years)', 'BMI (kg/m\u00b2)',
    'Systolic BP (mmHg)', 'Cholesterol (mg/dL)', 'Resting HR (bpm)'
]

for feat, pos, title in zip(features, feat_positions, feat_titles):
    ax = fig.add_subplot(pos)
    for cls, color, label in zip([0, 1], class_colors, class_labels):
        vals = df.loc[df['heart_disease'] == cls, feat]
        ax.hist(vals, bins=18, color=color, alpha=0.65,
                edgecolor='white', label=label, density=True)
    ax.set_xlabel(title, fontsize=9)
    ax.set_ylabel('Density', fontsize=9)
    ax.set_title(feat, fontsize=10)
    ax.legend(fontsize=7)

plt.suptitle('Step 1.2 — EDA: Class Distribution and Feature Histograms by Class',
             fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

print('Class distribution:')
for cls, count in class_counts.items():
    print(f'  Class {cls} ({class_labels[cls]}) : {count} patients ({count/N*100:.1f}%)')
print()
print('Feature means by class:')
print(df.groupby('heart_disease')[features].mean().round(2).to_string())

---
### Step 1.3 — Feature Normalization

Neural networks are **very sensitive to the scale of input features** (Session 25–26). Here is why:

- Each weight multiplies its input feature directly. If `sbp` ranges from 100–200 while `bmi` ranges from 18–38, the gradient updates for the `sbp` weights will be ~5× larger — causing the optimizer to overshoot for some weights and barely move for others.
- Normalization ensures that **every feature contributes equally at the start of training** and that gradient descent converges efficiently.

We use **StandardScaler** which applies:
$$z = \frac{x - \mu}{\sigma}$$

After scaling, each feature has mean ≈ 0 and standard deviation ≈ 1.

> **Important:** We fit the scaler **only on the training set** and then apply it to both train and test sets. Fitting on the full dataset would "leak" test-set statistics into training — a subtle but real form of data leakage.

In [ ]:
X = df[features].values
y = df['heart_disease'].values

# --- Before scaling ---
print('Feature statistics BEFORE scaling:')
for i, feat in enumerate(features):
    print(f'  {feat:<15}  mean={X[:, i].mean():7.2f}   std={X[:, i].std():6.2f}   '
          f'range=[{X[:, i].min():.1f}, {X[:, i].max():.1f}]')

# Train/test split first (before fitting scaler)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=9
)

# Fit scaler on training data only
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print()
print('Feature statistics AFTER scaling (training set):')
for i, feat in enumerate(features):
    print(f'  {feat:<15}  mean={X_train_sc[:, i].mean():7.4f}   std={X_train_sc[:, i].std():.4f}')

print()
print(f'Train samples : {X_train.shape[0]}  ({X_train.shape[0]/N:.0%})')
print(f'Test samples  : {X_test.shape[0]}   ({X_test.shape[0]/N:.0%})')

# --- Visual: before vs after scaling for age and sbp ---
fig, axes = plt.subplots(2, 2, figsize=(12, 7))
pairs = [
    (X_train[:, 0],    X_train_sc[:, 0],    'Age (years)',         'Age (scaled)'),
    (X_train[:, 2],    X_train_sc[:, 2],    'SBP (mmHg)',          'SBP (scaled)'),
]
for row_idx, (raw, scaled, raw_label, scaled_label) in enumerate(pairs):
    axes[row_idx, 0].hist(raw,    bins=20, color='steelblue', edgecolor='white', alpha=0.85)
    axes[row_idx, 0].set_xlabel(raw_label, fontsize=10)
    axes[row_idx, 0].set_ylabel('Count', fontsize=10)
    axes[row_idx, 0].set_title(f'Before Scaling: {raw_label}', fontsize=11)

    axes[row_idx, 1].hist(scaled, bins=20, color='darkorange', edgecolor='white', alpha=0.85)
    axes[row_idx, 1].set_xlabel(scaled_label, fontsize=10)
    axes[row_idx, 1].set_ylabel('Count', fontsize=10)
    axes[row_idx, 1].set_title(f'After Scaling: {scaled_label}', fontsize=11)

plt.suptitle('Step 1.3 — Feature Normalization: Before vs. After StandardScaler',
             fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

---
### Step 1.4 — Train / Test Split Summary

We already performed the split in Step 1.3. Here we confirm the class balance within each split to ensure neither set is accidentally dominated by one class.

A good split should preserve the approximate **40–60% positive rate** seen in the full dataset. If one class is severely underrepresented in the test set, our evaluation metrics will be unreliable.

In [ ]:
print('Train / Test Split — Class Balance Check')
print(f'  Total patients    : {N}')
print(f'  Training samples  : {len(y_train)}  ({len(y_train)/N:.0%})')
print(f'  Test samples      : {len(y_test)}   ({len(y_test)/N:.0%})')
print()

for split_name, split_y in [('Training', y_train), ('Test', y_test)]:
    n1 = split_y.sum()
    n0 = len(split_y) - n1
    print(f'  {split_name} set:')
    print(f'    Class 0 (No HD) : {n0} ({n0/len(split_y)*100:.1f}%)')
    print(f'    Class 1 (HD)    : {n1} ({n1/len(split_y)*100:.1f}%)')
    print()

print('Both splits reflect the overall class ratio — suitable for evaluation.')

---
### Step 1.5 — Build and Train the MLP

We use scikit-learn's `MLPClassifier`. Key parameters:

| Parameter | Value | Meaning |
|-----------|-------|--------|
| `hidden_layer_sizes` | `(100, 50)` | Two hidden layers: 100 neurons, then 50 neurons |
| `activation` | `'relu'` (default) | Rectified Linear Unit — fast, avoids vanishing gradients |
| `solver` | `'adam'` (default) | Adaptive Moment Estimation — efficient stochastic optimizer |
| `max_iter` | `500` | Maximum training epochs |
| `random_state` | `9` | Reproducibility |

**Backpropagation (Session 26):** During training, the model computes the **cross-entropy loss** between predicted probabilities and true labels, then propagates the gradient backward through each layer to update every weight:

$$\mathcal{L} = -\frac{1}{n} \sum_{i=1}^{n} \left[ y_i \log(\hat{p}_i) + (1 - y_i) \log(1 - \hat{p}_i) \right]$$

At each iteration (epoch), `model.loss_curve_` records the current training loss.

In [ ]:
model1 = MLPClassifier(
    hidden_layer_sizes=(100, 50),
    max_iter=500,
    random_state=9
)

model1.fit(X_train_sc, y_train)

print('=== Task 1 — MLP Network Summary ===')
print(f'  Architecture          : 5 → 100 → 50 → 1')
print(f'  Activation            : {model1.activation}')
print(f'  Solver                : {model1.solver}')
print(f'  Alpha (L2 reg)        : {model1.alpha}')
print(f'  Iterations run        : {model1.n_iter_}')
print(f'  Converged             : {model1.n_iter_ < model1.max_iter}')
print(f'  Final training loss   : {model1.loss_:.6f}')
print()

# Count parameters
n_params = sum(w.size for w in model1.coefs_) + sum(b.size for b in model1.intercepts_)
print(f'  Total trainable params: {n_params}')
print(f'    Input → Hidden 1    : 5 × 100 + 100 = {5*100 + 100} parameters')
print(f'    Hidden 1 → Hidden 2 : 100 × 50 + 50 = {100*50 + 50} parameters')
print(f'    Hidden 2 → Output   : 50 × 1 + 1   = {50*1 + 1} parameters')

---
### Step 1.6 — Evaluate: Accuracy + Confusion Matrix

For a binary classifier, the **confusion matrix** is the primary evaluation tool (Session 28). It shows the four possible outcomes for each prediction:

| | Predicted: No HD (0) | Predicted: HD (1) |
|-|---------------------|------------------|
| **Actual: No HD (0)** | True Negative (TN) | False Positive (FP) |
| **Actual: HD (1)** | False Negative (FN) | True Positive (TP) |

**Clinical meaning:**
- **TP** — Model correctly identifies a patient who will develop heart disease. Patient receives preventive care.
- **TN** — Model correctly clears a healthy patient. No unnecessary intervention.
- **FP** — Healthy patient is incorrectly flagged. Leads to unnecessary tests, anxiety, cost.
- **FN** — At-risk patient is missed. **Most dangerous clinically** — patient receives no intervention.

In cardiac risk screening, reducing **FN** (high recall) is typically more important than reducing **FP** (high precision).

In [ ]:
y_pred1 = model1.predict(X_test_sc)
acc1    = accuracy_score(y_test, y_pred1)
cm1     = confusion_matrix(y_test, y_pred1)

TN, FP, FN, TP = cm1.ravel()

print('=== Task 1 — Evaluation on Test Set ===')
print(f'  Accuracy  : {acc1:.4f}  ({acc1*100:.1f}%)')
print()
print(f'  Confusion Matrix:')
print(f'    TN = {TN:3d}  FP = {FP:3d}')
print(f'    FN = {FN:3d}  TP = {TP:3d}')
print()
print(f'  Clinical interpretation:')
print(f'    {TP} at-risk patients correctly identified (TP) — will receive preventive care')
print(f'    {TN} healthy patients correctly cleared (TN)')
print(f'    {FP} healthy patients unnecessarily flagged (FP) — extra cost and anxiety')
print(f'    {FN} at-risk patients missed (FN) — most dangerous: no intervention given')

# --- Heatmap ---
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(cm1, interpolation='nearest', cmap='Blues')
plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

cell_labels = [['TN', 'FP'], ['FN', 'TP']]
thresh = cm1.max() / 2.0
for i in range(2):
    for j in range(2):
        color = 'white' if cm1[i, j] > thresh else 'black'
        ax.text(j, i, f'{cell_labels[i][j]}\n{cm1[i, j]}',
                ha='center', va='center', fontsize=14,
                fontweight='bold', color=color)

ax.set_xticks([0, 1])
ax.set_yticks([0, 1])
ax.set_xticklabels(['Predicted: No HD (0)', 'Predicted: HD (1)'], fontsize=10)
ax.set_yticklabels(['Actual: No HD (0)', 'Actual: HD (1)'], fontsize=10)
ax.set_xlabel('Predicted Label', fontsize=11)
ax.set_ylabel('Actual Label', fontsize=11)
ax.set_title(f'Step 1.6 — Confusion Matrix\nAccuracy = {acc1*100:.1f}%', fontsize=12)
plt.tight_layout()
plt.show()

---
### Step 1.7 — Classification Report

The classification report (Session 28) provides three metrics per class:

$$\text{Precision} = \frac{TP}{TP + FP} \qquad \text{Recall} = \frac{TP}{TP + FN} \qquad F_1 = 2 \cdot \frac{\text{Precision} \times \text{Recall}}{\text{Precision} + \text{Recall}}$$

**Clinical meaning for Class 1 (Heart Disease):**
- **Precision:** Of all patients the model flags as high-risk, what fraction actually are?
  - Low precision → many unnecessary follow-ups and patient anxiety
- **Recall (Sensitivity):** Of all truly at-risk patients, what fraction did the model catch?
  - Low recall → dangerous missed cases — patients who needed help but were overlooked
- **F1-score:** Harmonic mean of precision and recall — useful when there is a class imbalance

In cardiac screening, **recall** is typically the more clinically critical metric.

In [ ]:
print('=== Task 1 — Classification Report ===')
print()
report = classification_report(
    y_test, y_pred1,
    target_names=['No Heart Disease (0)', 'Heart Disease (1)']
)
print(report)

# Manual calculation for transparency
TN, FP, FN, TP = confusion_matrix(y_test, y_pred1).ravel()
precision1 = TP / (TP + FP) if (TP + FP) > 0 else 0
recall1    = TP / (TP + FN) if (TP + FN) > 0 else 0
f1_1       = 2 * precision1 * recall1 / (precision1 + recall1) if (precision1 + recall1) > 0 else 0

print('Manual verification for Heart Disease class (1):')
print(f'  Precision = TP/(TP+FP) = {TP}/({TP}+{FP}) = {precision1:.4f}')
print(f'  Recall    = TP/(TP+FN) = {TP}/({TP}+{FN}) = {recall1:.4f}')
print(f'  F1-score  = 2*P*R/(P+R)                  = {f1_1:.4f}')
print()
print('Clinical interpretation:')
print(f'  The model catches {recall1*100:.1f}% of true heart-disease cases (recall).')
print(f'  When it flags a patient, it is correct {precision1*100:.1f}% of the time (precision).')

---
### Step 1.8 — Training Loss Curve

During training, the MLP minimizes the **cross-entropy loss** using the Adam optimizer. After calling `.fit()`, scikit-learn stores the loss at each epoch in `model.loss_curve_`.

**What to look for (Session 26 — backpropagation and convergence):**
- A **steep drop** early on indicates the model is learning quickly in the first few epochs
- The curve should **flatten** (plateau) as the model converges
- If the loss never flattens, `max_iter` may be too small
- If the loss drops then bounces back up on the validation set, the model is **overfitting**

The scikit-learn `MLPClassifier` records only the training loss, not a separate validation loss. Task 2 will address this limitation.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

epochs = np.arange(1, len(model1.loss_curve_) + 1)
ax.plot(epochs, model1.loss_curve_, color='steelblue', linewidth=2, label='Training Loss')

# Mark the convergence region (last 10% of iterations)
conv_start = int(len(epochs) * 0.9)
ax.axvspan(epochs[conv_start], epochs[-1], alpha=0.12, color='green',
           label='Convergence region')
ax.axhline(model1.loss_curve_[-1], color='firebrick', linestyle='--', linewidth=1.2,
           label=f'Final loss = {model1.loss_curve_[-1]:.4f}')

# Annotate steep drop
ax.annotate('Rapid learning\n(early epochs)',
            xy=(epochs[5], model1.loss_curve_[5]),
            xytext=(epochs[5] + max(epochs)*0.06, model1.loss_curve_[5] + 0.05),
            arrowprops=dict(arrowstyle='->', color='black'),
            fontsize=9)

ax.set_xlabel('Epoch (Iteration)', fontsize=11)
ax.set_ylabel('Cross-Entropy Loss', fontsize=11)
ax.set_title(f'Step 1.8 — Training Loss Curve  '
             f'(Architecture: 5→100→50→1, Epochs: {model1.n_iter_})',
             fontsize=12)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print('Loss curve summary:')
print(f'  Starting loss (epoch 1) : {model1.loss_curve_[0]:.6f}')
print(f'  Final loss    (epoch {model1.n_iter_}) : {model1.loss_curve_[-1]:.6f}')
print(f'  Total reduction         : {model1.loss_curve_[0] - model1.loss_curve_[-1]:.6f}')
print(f'  Converged in {model1.n_iter_} of {model1.max_iter} max iterations')
if model1.n_iter_ < model1.max_iter:
    print('  Status: CONVERGED — loss tolerance was reached before max_iter.')
else:
    print('  Status: DID NOT CONVERGE — consider increasing max_iter or adjusting learning rate.')

---
### Step 1.9 — Interactive Widget: Explore Architecture

This widget lets you experiment with the MLP architecture interactively without rewriting code.

**Parameters you control:**
- **Neurons Layer 1** — neurons in the first hidden layer
- **Neurons Layer 2** — neurons in the second hidden layer (set to 0 for a single-hidden-layer network)
- **Alpha (L2 reg)** — L2 regularization strength (larger alpha → simpler model)
- **Max Iterations** — maximum training epochs

**Experiments to try:**
1. Set Layer 2 = 0 and reduce Layer 1 to 5 — observe the loss curve with an underpowered network
2. Set Layer 1 = 200, Layer 2 = 100 — does adding more capacity help or cause overfitting?
3. Increase alpha to 1.0 — how does strong L2 regularization affect accuracy?
4. Reduce max_iter to 50 — does the network converge?

In [ ]:
def run_mlp_widget(neurons_l1, neurons_l2, alpha, max_iter):
    # Build architecture tuple
    if neurons_l2 > 0:
        architecture = (neurons_l1, neurons_l2)
        arch_str = f'5 → {neurons_l1} → {neurons_l2} → 1'
    else:
        architecture = (neurons_l1,)
        arch_str = f'5 → {neurons_l1} → 1'

    clf = MLPClassifier(
        hidden_layer_sizes=architecture,
        alpha=alpha,
        max_iter=max_iter,
        random_state=9
    )
    clf.fit(X_train_sc, y_train)

    y_pred_tr = clf.predict(X_train_sc)
    y_pred_te = clf.predict(X_test_sc)
    acc_tr = accuracy_score(y_train, y_pred_tr)
    acc_te = accuracy_score(y_test,  y_pred_te)
    gap    = acc_tr - acc_te
    cm_w   = confusion_matrix(y_test, y_pred_te)

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle(
        f'Architecture: {arch_str}  |  alpha={alpha}  |  max_iter={max_iter}',
        fontsize=12
    )

    # Loss curve
    ep = np.arange(1, len(clf.loss_curve_) + 1)
    axes[0].plot(ep, clf.loss_curve_, color='steelblue', linewidth=2)
    axes[0].set_xlabel('Epoch', fontsize=10)
    axes[0].set_ylabel('Cross-Entropy Loss', fontsize=10)
    axes[0].set_title(f'Loss Curve  (final={clf.loss_curve_[-1]:.4f}, '
                      f'epochs={clf.n_iter_})', fontsize=11)
    axes[0].grid(True, alpha=0.3)

    # Confusion matrix
    im = axes[1].imshow(cm_w, cmap='Blues', interpolation='nearest')
    plt.colorbar(im, ax=axes[1], fraction=0.046, pad=0.04)
    cell_lbls = [['TN', 'FP'], ['FN', 'TP']]
    thresh_w = cm_w.max() / 2.0
    for i in range(2):
        for j in range(2):
            color = 'white' if cm_w[i, j] > thresh_w else 'black'
            axes[1].text(j, i, f'{cell_lbls[i][j]}\n{cm_w[i, j]}',
                         ha='center', va='center', fontsize=13,
                         fontweight='bold', color=color)
    axes[1].set_xticks([0, 1])
    axes[1].set_yticks([0, 1])
    axes[1].set_xticklabels(['Pred: No HD', 'Pred: HD'], fontsize=10)
    axes[1].set_yticklabels(['Actual: No HD', 'Actual: HD'], fontsize=10)
    axes[1].set_title(f'Confusion Matrix  (Test Acc={acc_te*100:.1f}%)', fontsize=11)

    plt.tight_layout()
    plt.show()

    print(f'  Train accuracy : {acc_tr*100:.2f}%')
    print(f'  Test  accuracy : {acc_te*100:.2f}%')
    print(f'  Overfitting gap: {gap*100:.2f} percentage points', end='')
    if gap > 0.10:
        print('  <<< WARNING: possible overfitting (gap > 10 pp) >>>')
    elif gap < 0.0:
        print('  (test > train — unusual, may be noise with small test set)')
    else:
        print('  (acceptable gap)')


widgets.interact(
    run_mlp_widget,
    neurons_l1=widgets.IntSlider(
        min=5, max=200, step=5, value=100,
        description='Neurons Layer 1:',
        style={'description_width': 'initial'},
        layout=widgets.Layout(width='60%')
    ),
    neurons_l2=widgets.IntSlider(
        min=0, max=100, step=5, value=50,
        description='Neurons Layer 2 (0=single layer):',
        style={'description_width': 'initial'},
        layout=widgets.Layout(width='60%')
    ),
    alpha=widgets.FloatSlider(
        min=0.0001, max=1.0, step=0.01, value=0.0001,
        description='Alpha (L2 reg):',
        style={'description_width': 'initial'},
        layout=widgets.Layout(width='60%'),
        readout_format='.4f'
    ),
    max_iter=widgets.IntSlider(
        min=50, max=1000, step=50, value=500,
        description='Max Iterations:',
        style={'description_width': 'initial'},
        layout=widgets.Layout(width='60%')
    )
)

---
---
# TASK 2 — Deeper Network: Debugging and Improvement

A neural network that **memorizes** the training data rather than learning generalizable patterns will perform well in-house but fail on new patients. This phenomenon — **overfitting** — is one of the central challenges in applied machine learning (Session 28).

In Task 2 we will:
1. **Demonstrate** overfitting and underfitting by comparing three architectures
2. **Regularize** with L2 penalty to reduce overfitting
3. **Cross-validate** to get a robust estimate of generalization performance
4. **Plot the ROC curve** to evaluate threshold-independent discriminability
5. **Apply** the best model to new MRHA patients
6. **Explore** interactively with a widget

---
### Step 2.1 — Overfitting Demonstration

We compare three neural network architectures (Session 28 — bias–variance tradeoff):

| Model | Architecture | Expected behavior |
|-------|-------------|-------------------|
| **Underfitting** | (5,) | Too few neurons — cannot capture the signal |
| **Good fit** | (100, 50) | Balanced capacity — our baseline from Task 1 |
| **Overfitting** | (300, 300, 200) | Too many neurons — memorizes training set |

**Signs of overfitting:** train accuracy >> test accuracy  
**Signs of underfitting:** both train and test accuracy are low

The goal is to find an architecture where the **test accuracy is high** and the **gap between train and test is small**.

In [ ]:
architectures = [
    ((5,),          'Underfitting\n(5,)',          'steelblue'),
    ((100, 50),     'Good Fit\n(100,50)',           'seagreen'),
    ((300, 300, 200),'Overfitting\n(300,300,200)', 'firebrick'),
]

train_accs = []
test_accs  = []
model_names = []

for arch, label, color in architectures:
    clf = MLPClassifier(hidden_layer_sizes=arch, max_iter=1000, random_state=9)
    clf.fit(X_train_sc, y_train)
    tr_acc = accuracy_score(y_train, clf.predict(X_train_sc))
    te_acc = accuracy_score(y_test,  clf.predict(X_test_sc))
    train_accs.append(tr_acc)
    test_accs.append(te_acc)
    model_names.append(label)
    print(f'{label.replace(chr(10)," "):<30}  train={tr_acc*100:.1f}%  test={te_acc*100:.1f}%  '
          f'gap={( tr_acc - te_acc)*100:.1f} pp')

# --- Grouped bar chart ---
fig, ax = plt.subplots(figsize=(10, 6))
x = np.arange(len(architectures))
width = 0.35

bars_tr = ax.bar(x - width/2, [a*100 for a in train_accs],
                 width, label='Train Accuracy', color='steelblue', edgecolor='white')
bars_te = ax.bar(x + width/2, [a*100 for a in test_accs],
                 width, label='Test Accuracy',  color='darkorange', edgecolor='white')

for bar in bars_tr:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.4,
            f'{bar.get_height():.1f}%', ha='center', va='bottom', fontsize=9)
for bar in bars_te:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.4,
            f'{bar.get_height():.1f}%', ha='center', va='bottom', fontsize=9)

ax.set_xticks(x)
ax.set_xticklabels(model_names, fontsize=10)
ax.set_ylabel('Accuracy (%)', fontsize=11)
ax.set_ylim(50, 105)
ax.set_title('Step 2.1 — Underfitting / Good Fit / Overfitting Comparison', fontsize=12)
ax.legend(fontsize=10)
ax.axhline(50, color='grey', linestyle=':', linewidth=0.8, label='Random baseline')
plt.tight_layout()
plt.show()

---
### Step 2.2 — Regularization (L2 Penalty)

**L2 regularization** (also called **weight decay**) penalizes large weights by adding a term to the loss function (Session 28):

$$\mathcal{L}_{\text{regularized}} = \mathcal{L}_{\text{CE}} + \alpha \sum_{i,j} w_{ij}^2$$

- **Large alpha** → weights are pushed toward zero → simpler model → less overfitting but possibly underfitting
- **Small alpha** → almost no penalty → model can overfit freely

We compare three alpha values and visualize the resulting confusion matrices side by side.

| Alpha | Expected effect |
|-------|----------------|
| 0.0001 | Very weak regularization — baseline |
| 0.01 | Moderate regularization — often the sweet spot |
| 0.5 | Strong regularization — may underfit |

**Clinical framing:** Regularization is analogous to applying Occam's Razor in medicine — prefer the simpler explanation that fits the data, rather than an overly complex model that may not generalize to future patients.

In [ ]:
alpha_values = [0.0001, 0.01, 0.5]
alpha_labels = ['alpha=0.0001\n(very weak)', 'alpha=0.01\n(moderate)', 'alpha=0.5\n(strong)']
alpha_colors = ['steelblue', 'seagreen', 'darkorange']

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, alpha_val, label, color in zip(axes, alpha_values, alpha_labels, alpha_colors):
    clf = MLPClassifier(
        hidden_layer_sizes=(100, 50),
        alpha=alpha_val, max_iter=500, random_state=9
    )
    clf.fit(X_train_sc, y_train)
    y_pred_a  = clf.predict(X_test_sc)
    acc_a     = accuracy_score(y_test, y_pred_a)
    cm_a      = confusion_matrix(y_test, y_pred_a)
    TN_a, FP_a, FN_a, TP_a = cm_a.ravel()

    im = ax.imshow(cm_a, cmap='Blues', interpolation='nearest')
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    cell_lbls = [['TN', 'FP'], ['FN', 'TP']]
    thresh_a = cm_a.max() / 2.0
    for i in range(2):
        for j in range(2):
            c = 'white' if cm_a[i, j] > thresh_a else 'black'
            ax.text(j, i, f'{cell_lbls[i][j]}\n{cm_a[i, j]}',
                    ha='center', va='center', fontsize=12, fontweight='bold', color=c)
    ax.set_xticks([0, 1])
    ax.set_yticks([0, 1])
    ax.set_xticklabels(['Pred: No HD', 'Pred: HD'], fontsize=9)
    ax.set_yticklabels(['Actual: No HD', 'Actual: HD'], fontsize=9)
    ax.set_title(f'{label}\nAcc={acc_a*100:.1f}%  FN={FN_a}', fontsize=11)

plt.suptitle('Step 2.2 — Effect of L2 Regularization (alpha) on Confusion Matrix',
             fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

print('Comparison table:')
print(f'{"Alpha":<12} {"Accuracy":>10} {"FN (missed)":>13} {"FP (false alarm)":>18}')
print('-' * 56)
for alpha_val in alpha_values:
    clf = MLPClassifier(hidden_layer_sizes=(100, 50),
                        alpha=alpha_val, max_iter=500, random_state=9)
    clf.fit(X_train_sc, y_train)
    y_p = clf.predict(X_test_sc)
    cm_ = confusion_matrix(y_test, y_p)
    tn_, fp_, fn_, tp_ = cm_.ravel()
    acc_ = accuracy_score(y_test, y_p)
    print(f'{alpha_val:<12} {acc_*100:>9.1f}% {fn_:>13} {fp_:>18}')

---
### Step 2.3 — Cross-Validation with a Pipeline

A single train/test split gives only one estimate of performance — which may be lucky or unlucky depending on which patients ended up in the test set. **K-fold cross-validation** (Session 28) gives a more reliable estimate by averaging over K different splits.

**Why we use a Pipeline:**  
If we apply `StandardScaler` before splitting, we inadvertently use test-set statistics when computing the mean and std — a subtle form of **data leakage**. A `Pipeline` wraps the scaler and model together so that:
1. In each fold, the scaler is **fit only on the training fold**
2. The scaler is then **applied** to both train and validation folds
3. The model is trained on scaled training data and evaluated on scaled validation data

This mirrors the correct deployment process: fit the scaler on historical data, apply it to new patients.

In [ ]:
# Build pipeline: scaling + MLP in one object
pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('mlp',    MLPClassifier(
        hidden_layer_sizes=(100, 50),
        max_iter=500,
        random_state=9
    ))
])

# 5-fold cross-validation on the full dataset
cv_scores = cross_val_score(pipe, X, y, cv=5, scoring='accuracy')

print('=== Step 2.3 — 5-Fold Cross-Validation Results ===')
print()
for fold_idx, score in enumerate(cv_scores, start=1):
    bar = '#' * int(score * 40)
    print(f'  Fold {fold_idx}: {score*100:.2f}%  {bar}')
print()
print(f'  Mean accuracy : {cv_scores.mean()*100:.2f}%')
print(f'  Std deviation : {cv_scores.std()*100:.2f}%')
print(f'  95% CI approx : [{(cv_scores.mean() - 2*cv_scores.std())*100:.2f}%, '
      f'{(cv_scores.mean() + 2*cv_scores.std())*100:.2f}%]')
print()
print('Why Pipeline prevents data leakage:')
print('  In each of the 5 folds, StandardScaler is fit ONLY on the 4 training folds.')
print('  The validation fold is scaled with those parameters but never influences them.')
print('  This mirrors real deployment: scaler is trained on historical data only.')

# --- Bar chart of fold scores ---
fig, ax = plt.subplots(figsize=(8, 4))
fold_labels = [f'Fold {i}' for i in range(1, 6)]
bars = ax.bar(fold_labels, cv_scores * 100, color='steelblue',
              edgecolor='white', width=0.5)
ax.axhline(cv_scores.mean() * 100, color='firebrick', linestyle='--', linewidth=1.5,
           label=f'Mean = {cv_scores.mean()*100:.2f}%')
ax.fill_between(range(-1, 6),
                (cv_scores.mean() - cv_scores.std()) * 100,
                (cv_scores.mean() + cv_scores.std()) * 100,
                alpha=0.15, color='firebrick', label=f'±1 std ({cv_scores.std()*100:.2f}%)')
for bar, score in zip(bars, cv_scores):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
            f'{score*100:.1f}%', ha='center', va='bottom', fontsize=10)
ax.set_xlim(-0.5, 4.5)
ax.set_ylim(max(0, cv_scores.min()*100 - 5), min(100, cv_scores.max()*100 + 5))
ax.set_ylabel('Accuracy (%)', fontsize=11)
ax.set_title('Step 2.3 — 5-Fold Cross-Validation Accuracy per Fold', fontsize=12)
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

---
### Step 2.4 — ROC Curve and AUC

Accuracy depends on the chosen **decision threshold** (default 0.5 — predict HD if probability > 50%). The **Receiver Operating Characteristic (ROC) curve** shows how **True Positive Rate (Recall)** and **False Positive Rate** trade off as we vary this threshold from 0 to 1 (Session 27 — clinical applications).

$$TPR = \text{Recall} = \frac{TP}{TP+FN} \qquad FPR = \frac{FP}{FP+TN}$$

**Area Under the ROC Curve (AUC):**
- AUC = 1.0 → perfect classifier
- AUC = 0.5 → random classifier (diagonal line)
- AUC = 0.7–0.8 → acceptable for a screening tool
- AUC = 0.8–0.9 → excellent for a screening tool

**Clinical framing:** A clinician can choose the operating point on the ROC curve:
- Move up-left → higher sensitivity (catch more cases, accept more false alarms) → screening
- Move down-right → higher specificity (fewer false alarms, accept some missed cases) → confirmation

This flexibility is why AUC is preferred over accuracy alone in medical AI evaluation.

In [ ]:
# Use predict_proba to get probability scores for the positive class
y_prob1 = model1.predict_proba(X_test_sc)[:, 1]

fpr, tpr, thresholds = roc_curve(y_test, y_prob1)
roc_auc = auc(fpr, tpr)

# Find the threshold closest to default (0.5)
idx_05 = np.argmin(np.abs(thresholds - 0.5))

fig, ax = plt.subplots(figsize=(8, 7))

ax.plot(fpr, tpr, color='steelblue', linewidth=2.5,
        label=f'MLP Classifier  (AUC = {roc_auc:.3f})')
ax.plot([0, 1], [0, 1], 'k--', linewidth=1.2, label='Random classifier (AUC = 0.5)')

# Mark the default operating point
ax.scatter(fpr[idx_05], tpr[idx_05], color='firebrick', s=120, zorder=5,
           label=f'Default threshold (0.5)\nTPR={tpr[idx_05]:.2f}, FPR={fpr[idx_05]:.2f}')

# Shade AUC region
ax.fill_between(fpr, tpr, alpha=0.12, color='steelblue')

ax.set_xlabel('False Positive Rate  (1 - Specificity)', fontsize=12)
ax.set_ylabel('True Positive Rate  (Sensitivity / Recall)', fontsize=12)
ax.set_title(f'Step 2.4 — ROC Curve\nAUC = {roc_auc:.3f}', fontsize=13)
ax.legend(fontsize=10, loc='lower right')
ax.set_xlim([0, 1])
ax.set_ylim([0, 1.02])
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f'AUC = {roc_auc:.4f}')
print()
if roc_auc >= 0.9:
    interpretation = 'outstanding'
elif roc_auc >= 0.8:
    interpretation = 'excellent'
elif roc_auc >= 0.7:
    interpretation = 'acceptable'
else:
    interpretation = 'poor'
print(f'Clinical interpretation: AUC = {roc_auc:.3f} is {interpretation} for a screening tool.')
print(f'At the default threshold (0.50):')
print(f'  True Positive Rate (Sensitivity) : {tpr[idx_05]:.3f} — the model catches {tpr[idx_05]*100:.1f}% of true cases')
print(f'  False Positive Rate              : {fpr[idx_05]:.3f} — {fpr[idx_05]*100:.1f}% of healthy patients are incorrectly flagged')

---
### Step 2.5 — Clinical Predictions for New MRHA Patients

The MRHA clinical team has referred 5 new patients for assessment. Using the trained model, we will:

1. Prepare their feature vectors
2. Apply the same `StandardScaler` (already fit on the training data) — **never re-fit the scaler**
3. Obtain predicted **class labels** (0/1) and **probability scores**
4. Interpret the results in clinical terms

**Session 27 — Clinical applications:** Probability scores are more useful than hard labels. A physician may wish to intervene at a lower threshold (e.g., 30% probability) for a patient with multiple modifiable risk factors, while a hard label would show `0` regardless.

> **Important:** In a real deployment, we would use the full Pipeline (scaler + model) trained on all available data, not just the 80% training split. Here we use the already-trained model and scaler for demonstration.

In [ ]:
# New patient records: [age, bmi, sbp, cholesterol, resting_hr]
new_patients_raw = np.array([
    [45, 22.1, 118, 185, 68],   # Patient A: young, healthy profile
    [62, 30.5, 155, 260, 88],   # Patient B: older, overweight, high BP and cholesterol
    [55, 27.0, 135, 215, 74],   # Patient C: mid-range risk
    [38, 19.2, 108, 160, 62],   # Patient D: young, low risk
    [70, 35.8, 175, 300, 98],   # Patient E: elderly, multiple elevated factors
])

patient_ids = ['Patient A', 'Patient B', 'Patient C', 'Patient D', 'Patient E']

# Scale using the already-fitted scaler (fit on training data)
new_patients_sc = scaler.transform(new_patients_raw)

# Predict
predictions = model1.predict(new_patients_sc)
probabilities = model1.predict_proba(new_patients_sc)[:, 1]

print('=== Step 2.5 — Clinical Predictions for New MRHA Patients ===')
print()
print(f'{"Patient":<12} {"Age":>4} {"BMI":>5} {"SBP":>5} {"Chol":>5} {"HR":>4} '
      f'{"P(HD)":>8} {"Prediction":<20} {"Clinical Flag"}')
print('-' * 95)

flag_colors = {0: 'LOW RISK', 1: 'HIGH RISK — refer for further evaluation'}
for pid, raw, pred, prob in zip(patient_ids, new_patients_raw, predictions, probabilities):
    flag = flag_colors[pred]
    pred_label = 'Heart Disease' if pred == 1 else 'No Heart Disease'
    print(f'{pid:<12} {int(raw[0]):>4} {raw[1]:>5.1f} {raw[2]:>5.1f} '
          f'{raw[3]:>5.1f} {raw[4]:>4.1f} '
          f'{prob:>7.1%} {pred_label:<20} {flag}')

# --- Bar chart of probabilities ---
fig, ax = plt.subplots(figsize=(10, 5))
bar_colors = ['firebrick' if p == 1 else 'steelblue' for p in predictions]
bars = ax.bar(patient_ids, probabilities * 100, color=bar_colors, edgecolor='white', width=0.5)
ax.axhline(50, color='black', linestyle='--', linewidth=1.5, label='Default threshold (50%)')
ax.axhspan(0,  30, alpha=0.07, color='steelblue', label='Low risk zone')
ax.axhspan(50, 100, alpha=0.07, color='firebrick', label='High risk zone')
for bar, prob in zip(bars, probabilities):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
            f'{prob*100:.1f}%', ha='center', va='bottom', fontsize=11, fontweight='bold')
ax.set_ylabel('Predicted Probability of Heart Disease (%)', fontsize=11)
ax.set_title('Step 2.5 — Heart Disease Risk Probability: New MRHA Patients', fontsize=12)
ax.set_ylim(0, 110)
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

---
### Step 2.6 — Interactive Widget: Architecture and Regularization Explorer

This widget extends the one from Task 1 (Step 1.9). It shows **both the loss curve and the confusion matrix side by side**, and explicitly reports the train/test accuracy gap with an overfitting warning.

**Suggested experiments:**
1. Start with Layer 1=100, Layer 2=50, alpha=0.0001 — this is the baseline from Task 1
2. Increase Layer 1 to 300 and Layer 2 to 200 — does the test accuracy improve or worsen?
3. After seeing overfitting, raise alpha to 0.1 — does it close the train/test gap?
4. Try alpha=0.5 — is there a point where regularization hurts too much?
5. Set max_iter=100 — can you see an unconverged loss curve?

In [ ]:
def run_task2_widget(hidden_layer_1, hidden_layer_2, alpha, max_iter):
    if hidden_layer_2 > 0:
        architecture = (hidden_layer_1, hidden_layer_2)
        arch_str = f'5 → {hidden_layer_1} → {hidden_layer_2} → 1'
    else:
        architecture = (hidden_layer_1,)
        arch_str = f'5 → {hidden_layer_1} → 1'

    clf = MLPClassifier(
        hidden_layer_sizes=architecture,
        alpha=alpha,
        max_iter=max_iter,
        random_state=9
    )
    clf.fit(X_train_sc, y_train)

    y_pred_tr2 = clf.predict(X_train_sc)
    y_pred_te2 = clf.predict(X_test_sc)
    acc_tr2    = accuracy_score(y_train, y_pred_tr2)
    acc_te2    = accuracy_score(y_test,  y_pred_te2)
    gap2       = acc_tr2 - acc_te2
    cm_2       = confusion_matrix(y_test, y_pred_te2)
    TN2, FP2, FN2, TP2 = cm_2.ravel()

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle(
        f'Architecture: {arch_str}  |  alpha={alpha}  |  max_iter={max_iter}',
        fontsize=12
    )

    # Loss curve
    ep2 = np.arange(1, len(clf.loss_curve_) + 1)
    axes[0].plot(ep2, clf.loss_curve_, color='seagreen', linewidth=2)
    axes[0].axhline(clf.loss_curve_[-1], color='firebrick', linestyle='--', linewidth=1,
                    label=f'Final loss={clf.loss_curve_[-1]:.4f}')
    axes[0].set_xlabel('Epoch', fontsize=10)
    axes[0].set_ylabel('Cross-Entropy Loss', fontsize=10)
    axes[0].set_title(f'Loss Curve  (epochs={clf.n_iter_})', fontsize=11)
    axes[0].legend(fontsize=9)
    axes[0].grid(True, alpha=0.3)

    # Confusion matrix
    im2 = axes[1].imshow(cm_2, cmap='Blues', interpolation='nearest')
    plt.colorbar(im2, ax=axes[1], fraction=0.046, pad=0.04)
    cell_lbls2 = [['TN', 'FP'], ['FN', 'TP']]
    thresh2 = cm_2.max() / 2.0
    for i in range(2):
        for j in range(2):
            c2 = 'white' if cm_2[i, j] > thresh2 else 'black'
            axes[1].text(j, i, f'{cell_lbls2[i][j]}\n{cm_2[i, j]}',
                         ha='center', va='center', fontsize=13, fontweight='bold', color=c2)
    axes[1].set_xticks([0, 1])
    axes[1].set_yticks([0, 1])
    axes[1].set_xticklabels(['Pred: No HD', 'Pred: HD'], fontsize=10)
    axes[1].set_yticklabels(['Actual: No HD', 'Actual: HD'], fontsize=10)
    axes[1].set_title(f'Confusion Matrix  (Test Acc={acc_te2*100:.1f}%)', fontsize=11)

    plt.tight_layout()
    plt.show()

    print(f'  Train accuracy : {acc_tr2*100:.2f}%')
    print(f'  Test  accuracy : {acc_te2*100:.2f}%')
    print(f'  Gap (train-test): {gap2*100:.2f} pp', end='')
    if gap2 > 0.10:
        print('  <<< WARNING: possible overfitting (gap > 10 pp). Try increasing alpha. >>>')
    elif gap2 < -0.02:
        print('  (test > train — possible underfitting or small-sample noise)')
    else:
        print('  (acceptable generalization gap)')
    print(f'  Missed cases (FN): {FN2}  |  False alarms (FP): {FP2}')


widgets.interact(
    run_task2_widget,
    hidden_layer_1=widgets.IntSlider(
        min=5, max=300, step=5, value=100,
        description='Hidden Layer 1:',
        style={'description_width': 'initial'},
        layout=widgets.Layout(width='60%')
    ),
    hidden_layer_2=widgets.IntSlider(
        min=0, max=200, step=5, value=50,
        description='Hidden Layer 2 (0=single):',
        style={'description_width': 'initial'},
        layout=widgets.Layout(width='60%')
    ),
    alpha=widgets.FloatSlider(
        min=0.0001, max=1.0, step=0.01, value=0.0001,
        description='Alpha (L2 reg):',
        style={'description_width': 'initial'},
        layout=widgets.Layout(width='60%'),
        readout_format='.4f'
    ),
    max_iter=widgets.IntSlider(
        min=50, max=1000, step=50, value=500,
        description='Max Iterations:',
        style={'description_width': 'initial'},
        layout=widgets.Layout(width='60%')
    )
)

---
---
## Conclusions

This notebook built a complete neural network classification pipeline on a realistic cardiac dataset:

| | Task 1 | Task 2 |
|-|--------|--------|
| **Model** | MLP (100, 50) | MLP with regularization and cross-validation |
| **Input features** | 5 clinical variables | 5 clinical variables |
| **Target** | Heart disease (0/1) | Heart disease (0/1) |
| **Key concept** | MCP neuron → forward pass → backpropagation | Overfitting → L2 regularization → CV → ROC |
| **Sessions covered** | 25, 26 | 27, 28 |

**Key takeaways:**
- Feature normalization is *critical* for neural networks — without it, training is unreliable
- The confusion matrix reveals clinically important distinctions (FP vs FN) that accuracy hides
- A Pipeline prevents data leakage during cross-validation
- AUC summarizes classifier performance across all thresholds — more informative than a single accuracy number
- Regularization (alpha) trades model flexibility for generalization ability

---
## Reflection Questions

Write your answers in the Markdown cells below each question.

---

**Q1.** In Step 1.1, the AND gate and OR gate have the same weights ($w_1 = w_2 = 1$) but different thresholds. Explain in one sentence why changing the threshold alone is enough to switch between AND and OR logic.

*Answer:* ...

---

**Q2.** In Step 1.3, we scaled the features *after* the train/test split, fitting the scaler only on the training data. What would go wrong if we had scaled the *entire* dataset before splitting?

*Answer:* ...

---

**Q3.** The confusion matrix in Step 1.6 has both False Positives and False Negatives. From a clinical perspective, which type of error is more dangerous in cardiac risk screening, and why?

*Answer:* ...

---

**Q4.** Based on the overfitting demonstration in Step 2.1, describe what happens to the train accuracy and test accuracy as the network becomes very large (overfitting regime). Why does this happen?

*Answer:* ...

---

**Q5.** In Step 2.4, the ROC curve shows that the model can operate at different sensitivity/specificity tradeoffs. If MRHA wants to use this model for **mass population screening** (where catching every case is essential), should they move the threshold higher or lower than 0.5? What is the cost of this decision?

*Answer:* ...

---

**Q6. (Ethics)** MRHA proposes deploying this neural network as an automated triage tool: patients with predicted probability > 60% are immediately placed on a high-priority medication list without physician review. Identify at least **two specific risks** of this approach and propose a safeguard for each.

*Answer:* ...